In [1]:
import gymnasium as gym
import numpy as np
import subprocess
import re
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

In [ ]:
import gymnasium as gym
import numpy as np
import subprocess
import re

class CongestionEnv(gym.Env):
    def __init__(self):
        super().__init__()
        
        # state: throughput, delay
        self.observation_space = gym.spaces.Box(low=-1, high=1.0, shape=(5,), dtype=np.float32)
        
        # actions: decrease, same, increase window
        self.action_space = gym.spaces.Discrete(3)

        self.step_count = 0
        self.last_delay = 0
        
        self.window_sizes = [64, 128, 256, 512, 1024, 2048, 4096]  # KB
        self.idx = 2  # start at 256KB

    def step(self, action):
        # ----adjust window index
        self.step_count+=1

        # ----map actions
        action = int(action)
        action_map = {0: "↓ DECREASE", 1: "→ SAME", 2: "↑ INCREASE"}
        if action == 0:
            self.idx = max(0, self.idx - 1)
        elif action == 2:
            self.idx = min(len(self.window_sizes)-1, self.idx + 1)

        window = self.window_sizes[self.idx]

        # ---- get metrics
        throughput = self.get_throughput(window)
        delay = self.get_delay()
        loss = getattr(self, "last_loss", 0)

        #---- simulate congestion
        # if window >= 512:
        #     delay += (window / 1024) * np.random.uniform(5, 20)

        # if window >= 512:
        #     loss = np.random.binomial(1, (window / 1024) * 0.3)
        # else:
        #     loss = 0
        self.last_loss = loss

        #---- compute trends
        delta_delay = delay - self.last_delay
        self.last_delay = delay

        # ----Normalize values
        norm_throughput = min(throughput / 10, 1.0)
        
        norm_delay = (delay - 100) / 50
        norm_delay = np.clip(norm_delay, 0, 1)
        norm_delta_delay = np.clip(delta_delay / 50, -1, 1)

        norm_loss = min(loss / 5, 1.0)

        norm_cwnd = self.idx / (len(self.window_sizes) - 1)

        

        #---- reward

        # reward = (1.5 * norm_throughput- 1.0 * norm_delay- 1.5 * norm_loss- 0.3 * abs(norm_delta_delay))
        reward = (3.0 * norm_throughput- 1.5 * norm_delay- 0.05 * norm_loss - 0.2 * abs(norm_delta_delay))

        #---- extra penalties
        if delta_delay > 30:
            reward -= 0.01

        reward += 0.05 * norm_cwnd
        

        if action != self.last_action:
            reward -= 0.03
        self.last_action = action

        reward -= 0.005 * abs(action - 1)

        reward += 0.01 * np.random.rand()
            
        #----    Normalized state
        state = np.array([norm_throughput,norm_delay,norm_loss,norm_cwnd,norm_delta_delay], dtype=np.float32)

        #---- logging & printing
        # if not hasattr(self, "history"):
        #     self.history = []

        self.history.append((window, throughput, delay, loss, reward))

        print(
            f"Step {self.step_count} | {action_map[action]} | "
            f"CWND: {window} KB | "
            f"Thr: {throughput:.2f} | Delay: {delay:.2f} | Loss: {loss}"
        )

        return state, reward, False, False, {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.idx = 2
        self.step_count = 0
        self.last_delay = 0
        self.last_loss = 0
        self.last_action = 1  # assume "SAME"
        self.history = []
        return np.array([0.0, 0.0, 0.0, 0.0, 0.0], dtype=np.float32), {}

    def get_throughput(self, window):
        cmd = [
            "iperf3",
            "-c", "10.0.0.1",
            "-t", "8",
            "-w", f"{window}K"
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)
        output = result.stdout

        # throughput
        throughput_match = re.search(r'(\d+\.?\d*)\s*Mbits/sec.*sender', output)

        # retransmissions (loss)
        loss_match = re.search(r'(\d+)\s+sender', output)

        throughput = float(throughput_match.group(1)) if throughput_match else 0
        loss = int(loss_match.group(1)) if loss_match else 0

        self.last_loss = loss  # IMPORTANT

        return throughput

    def get_delay(self):
        cmd = ["ping", "-c", "3", "10.0.0.1"]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=5)
        output = result.stdout

        # extract avg RTT
        match = re.search(r'rtt .* = .*?/([\d\.]+)/', output)
        if match:
            return float(match.group(1))

        return 0
    


#iperf3 -c 10.0.0.1 -t 10    throughput
#ping -c 5 10.0.0.1     delay



In [ ]:
from stable_baselines3 import PPO
from env import CongestionEnv
import numpy as np

env = CongestionEnv()

# model = PPO("MlpPolicy", env, verbose=1, n_steps=8, batch_size=8)
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    n_steps=64,
    batch_size=32,
    gamma=0.99,
)


model.learn(total_timesteps=100)
np.save("history.npy", env.history)

# model.save("cc_rl_model")
# model.save("cc_rl_model_2")
model.save("cc_rl_model_3")



In [ ]:
from stable_baselines3 import PPO
from env import CongestionEnv


import matplotlib.pyplot as plt


env = CongestionEnv()
model = PPO.load("cc_rl_model_3", env=env)

obs, _ = env.reset()

total_throughput = 0
total_delay = 0
steps = 30

for _ in range(steps):
    action, _ = model.predict(obs)
    action = int(action)
    obs, reward, terminated, truncated, _ = env.step(action)

    throughput = obs[0] * 10
    delay = obs[1] * 50 + 100

    total_throughput += throughput
    total_delay += delay

    loss = getattr(env, "last_loss", 0)

    window = env.window_sizes[env.idx]

    action_map = {0: "↓ DECREASE", 1: "→ SAME", 2: "↑ INCREASE"}

    print(
        f"CWND: {window} KB | "
        f"Action: {action_map[action]} | "
        f"Throughput: {throughput:.2f} | "
        f"Delay: {delay:.2f} | "
        f"Loss: {loss}"
    )


history = env.history

if len(history) == 0:
    print("No data collected!")


windows = [h[0] for h in history]
throughputs = [h[1] for h in history]
delays = [h[2] for h in history]
losses = [h[3] for h in history]
rewards = [h[4] for h in history]

plt.figure(figsize=(12,8))

plt.subplot(4,1,1)
plt.plot(windows)
plt.title("CWND (KB)")

plt.subplot(4,1,2)
plt.plot(throughputs)
plt.title("Throughput (Mbps)")

plt.subplot(4,1,3)
plt.plot(delays)
plt.title("Delay (ms)")

plt.subplot(4,1,4)
plt.plot(losses)
plt.title("Loss")

plt.tight_layout()
plt.show()

print("RL Avg Throughput:", total_throughput / steps)
print("RL Avg Delay:", total_delay / steps)
